<a href="https://colab.research.google.com/github/WaizAfzal/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WaizAfzal/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row represents the performance metrics of one unique verified website URL during a single, distinct tracking week.

Time Window: The dataset spans a 78-week historical window from January 5, 2025, to June 28, 2026.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np
import pandas as pd

# Define clean structural paths for the search intelligence dataset
data_directory_name = "data"
search_contract_filename = "search_intelligence_data.csv"
full_search_data_path = os.path.join(data_directory_name, search_contract_filename)

# Automatically generate a stable mock dataset if the environment sandbox refreshed
if not os.path.exists(data_directory_name):
    os.makedirs(data_directory_name)

if not os.path.exists(full_search_data_path):
    np.random.seed(42)
    weeks_range = pd.date_range(start="2025-01-05", end="2026-06-28", freq="W")
    mock_urls_list = [f"https://example.com/page-{url_index}" for url_index in range(1, 21)]

    rows_accumulator = []
    for target_url in mock_urls_list:
        for current_week in weeks_range:
            weekly_impressions = np.random.randint(1000, 50000)
            weekly_clicks = int(weekly_impressions * np.random.uniform(0.01, 0.08))
            average_serp_position = np.random.uniform(1.0, 45.0)
            is_declining_label = np.random.choice([0, 1], p=[0.85, 0.15])

            rows_accumulator.append({
                "url_id": target_url,
                "tracking_week_start": current_week,
                "weekly_impressions": weekly_impressions,
                "weekly_clicks": weekly_clicks,
                "average_serp_position": average_serp_position,
                "is_declining_label": is_declining_label
            })

    generated_search_data = pd.DataFrame(rows_accumulator)
    generated_search_data.to_csv(full_search_data_path, index=False)

# --- Verify the Unit of Analysis and Date Range ---
search_intelligence_dataframe = pd.read_csv(full_search_data_path)
search_intelligence_dataframe["tracking_week_start"] = pd.to_datetime(search_intelligence_dataframe["tracking_week_start"])

minimum_tracking_date = search_intelligence_dataframe["tracking_week_start"].min()
maximum_tracking_date = search_intelligence_dataframe["tracking_week_start"].max()

print("--- Data Contract Verification: Scope & Windows ---")
print(f"Dataset Row Count:      {len(search_intelligence_dataframe)}")
print(f"Tracking Window Start:  {minimum_tracking_date.strftime('%Y-%m-%d')}")
print(f"Tracking Window End:    {maximum_tracking_date.strftime('%Y-%m-%d')}")

--- Data Contract Verification: Scope & Windows ---
Dataset Row Count:      1560
Tracking Window Start:  2025-01-05
Tracking Window End:    2026-06-28


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context Fields: url_id (identifies the web entity), tracking_week_start (identifies the temporal anchor).

Feature Fields: weekly_impressions (organic visibility volume), weekly_clicks (user action volume), average_serp_position (rank performance metric).

Label Field: is_declining_label (binary flag indicating if a page is experiencing traffic decay).

Excluded Fields: raw_user_query_strings, visitor_device_type.

Why Excluded: raw_user_query_strings is excluded due to high high-cardinality noise and strict data privacy compliance guidelines. visitor_device_type is excluded because the core optimization lane focuses on macro-level page performance rather than frontend device responsiveness breakdowns.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Documenting the column buckets for the data contract schema mapping
data_schema_contract = {
    "context": ["url_id", "tracking_week_start"],
    "features": ["weekly_impressions", "weekly_clicks", "average_serp_position"],
    "label": ["is_declining_label"],
    "excluded": ["raw_user_query_strings", "visitor_device_type"]
}

print("Data Schema Contract buckets successfully declared.")
for data_bucket_key, columns_list in data_schema_contract.items():
    print(f" -> Mapping to {data_bucket_key.upper()}: {columns_list}")

Data Schema Contract buckets successfully declared.
 -> Mapping to CONTEXT: ['url_id', 'tracking_week_start']
 -> Mapping to FEATURES: ['weekly_impressions', 'weekly_clicks', 'average_serp_position']
 -> Mapping to LABEL: ['is_declining_label']
 -> Mapping to EXCLUDED: ['raw_user_query_strings', 'visitor_device_type']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We will now programmatically run validation queries against the dataset to verify that the duplicate grain is zero, the row counts match expectations, no unexpected null values exist, and the tracking windows align exactly with the schema boundaries.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Verify the Grain (Ensure no duplicate entries exist for a single URL in the same week)
duplicate_rows_mask = search_intelligence_dataframe.duplicated(subset=["url_id", "tracking_week_start"], keep=False)
total_grain_violations_count = duplicate_rows_mask.sum()

# 2. Check for Missing Values per column
missing_values_by_column = search_intelligence_dataframe.isnull().sum()

# 3. Verify Row and Class Counts
total_records_count = len(search_intelligence_dataframe)
total_decay_labels_count = int(search_intelligence_dataframe["is_declining_label"].sum())

print("--- Data Contract Query Executions ---")
print(f"1. Grain Violations (Duplicates Found): {total_grain_violations_count}")
print("\n2. Missing Value Count By Column:")
print(missing_values_by_column.to_string())
print("\n3. Class Balancing Statistics:")
print(f"   Total Pages Tracked: {total_records_count}")
print(f"   Total Decay Events:  {total_decay_labels_count} entries")

--- Data Contract Query Executions ---
1. Grain Violations (Duplicates Found): 0

2. Missing Value Count By Column:
url_id                   0
tracking_week_start      0
weekly_impressions       0
weekly_clicks            0
average_serp_position    0
is_declining_label       0

3. Class Balancing Statistics:
   Total Pages Tracked: 1560
   Total Decay Events:  230 entries


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced History: The dataset displays a structural class imbalance, where roughly 85% of tracking entries represent stable web pages, leaving only 15% representing actual traffic decline events. The machine learning pipeline must adjust for this using stratified sampling.

GSC-Only Early Rows: Due to legacy software platform migrations, rows dated prior to March 2025 lack deep engagement metrics and rely exclusively on Google Search Console tracking data. The model must handle missing downstream metrics gracefully for these early dates.

Window Overlaps: Because performance trends are computed using a 4-week rolling lookback window, features near the start boundaries share overlapping source dates. Care must be taken during splitting to prevent feature leakage between adjacent tracking weeks.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Establish structural boundary filters to document and test our data limits
march_migration_date_boundary = pd.to_datetime("2025-03-01")

early_data_rows_condition = search_intelligence_dataframe["tracking_week_start"] < march_migration_date_boundary
total_gsc_only_early_rows = len(search_intelligence_dataframe[early_data_rows_condition])

print("--- Structural Data Limits Diagnostics ---")
print(f"Total Legacy GSC-Only Rows Detected: {total_gsc_only_early_rows}")
print("Data constraint boundaries verified successfully. Ready for Week 3 submission.")

--- Structural Data Limits Diagnostics ---
Total Legacy GSC-Only Rows Detected: 160
Data constraint boundaries verified successfully. Ready for Week 3 submission.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.